# Threshold based cell type assignment

## Defining thresholds 

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from anndata import AnnData
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from matplotlib.backends.backend_pdf import PdfPages
from scipy.optimize import fsolve

# Load data from CSV
csv_path = r"//IHOPE26_LN_InstanSeq_filtered_outliers.csv"
dataframe_filtered_normalized = pd.read_csv(csv_path)

# Define reliable data columns
reliable_data = ['cell_id', 'size', 'x', 'y', 'AQP4', 'CD3', 'CD31', 'CD45', 'CD163', 'Claudin5', 'FoxA2', 'GFAP', 
                 'HLA-DR', 'Ki67', 'MAP2', 'NEFL', 'Nes', 'NeuN', 'Olig2', 'P2Y12', 'PRKN', 'SNCA', 'Sox2', 'Sox9', 
                 'Syp', 'TMEM119', 'TUBB3', 'Ubiquitin', 'Vim']
dataframe_filtered_normalized = dataframe_filtered_normalized[reliable_data]

# Define metadata and marker data columns
metadata = ['cell_id', 'size', 'x', 'y']
marker_data = list(set(reliable_data) - set(metadata))

# Extract coordinates
coordinates = np.array(dataframe_filtered_normalized[['x', 'y']])

# Prepare AnnData object
meta = dataframe_filtered_normalized[metadata]
meta.index = meta.index.astype(str)

only_num = dataframe_filtered_normalized.drop(metadata, axis=1)
only_num.columns = only_num.columns.astype(str)
adata_all = AnnData(X=only_num.values, obs=meta)
adata_all.obsm["coord_xy"] = coordinates
adata_all.var_names = marker_data
adata_all.var["markers"] = only_num.columns.tolist()

# Function to fit two Gaussians and determine the threshold
def find_threshold(data, marker, save_path):
    data = data[~np.isnan(data)]  # Remove NaNs
    data = data.reshape(-1, 1)
    
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(data)
    means = gmm.means_.flatten()
    covariances = np.sqrt(gmm.covariances_).flatten()
    weights = gmm.weights_
    
    # Sort means to ensure the lower intensity peak is first
    sorted_indices = np.argsort(means)
    means, covariances, weights = means[sorted_indices], covariances[sorted_indices], weights[sorted_indices]
    
    # Define Gaussian functions
    def gaussian(x, mean, std, weight):
        return weight * norm.pdf(x, mean, std)
    
    # Function to find intersection
    def intersection(x):
        return gaussian(x, means[0], covariances[0], weights[0]) - gaussian(x, means[1], covariances[1], weights[1])
    
    # Solve for intersection
    threshold_guess = (means[0] + means[1]) / 2
    threshold = fsolve(intersection, threshold_guess)[0]
    
    # Generate x values for plotting
    x_values = np.linspace(data.min(), data.max(), 1000)
    
    # Compute Gaussian distributions
    gaussian1 = gaussian(x_values, means[0], covariances[0], weights[0])
    gaussian2 = gaussian(x_values, means[1], covariances[1], weights[1])
    
    # Plot histogram and fitted Gaussians
    plt.figure(figsize=(6, 4))
    plt.hist(data, bins=50, density=True, alpha=0.6, color='gray', label='Histogram')
    plt.plot(x_values, gaussian1, label='Gaussian 1', linestyle='dashed')
    plt.plot(x_values, gaussian2, label='Gaussian 2', linestyle='dashed')
    plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.2f}')
    plt.title(f"Marker: {marker}")
    plt.xlabel("Expression Level")
    plt.ylabel("Density")
    plt.legend()
    plt.savefig(os.path.join(save_path, f"{marker}_threshold_plot.png"))
    plt.close()
    
    return threshold

# Compute and save thresholds
thresholds = {}
output_dir = r"C:\Users\ivasu\Desktop\PDDLBD_PARI_Pilot_OptimisationAnalaysis\PARI4_PD_20X4X2048_individual\rollingball-r10-minmax-pixelfilteirng\rollingball-r10-minmax-pixelfiltering-pipexseq-myANALYSIS\Tresholding"
os.makedirs(output_dir, exist_ok=True)

pdf_path = os.path.join(output_dir, "threshold_histograms.pdf")
with PdfPages(pdf_path) as pdf:
    for marker in marker_data:
        thresholds[marker] = find_threshold(only_num[marker].values, marker, output_dir)
        plt.figure(figsize=(6, 4))
        plt.hist(only_num[marker].values, bins=50, density=True, alpha=0.6, color='gray', label='Histogram')
        plt.axvline(thresholds[marker], color='red', linestyle='--', label=f'Threshold: {thresholds[marker]:.2f}')
        plt.title(f"Marker: {marker}")
        plt.xlabel("Expression Level")
        plt.ylabel("Density")
        plt.legend()
        pdf.savefig()
        plt.close()

print(f"PDF with histograms saved at {pdf_path}")

# Save AnnData object
adata_save_path = r"C:\Users\ivasu\Desktop\PDDLBD_PARI_Pilot_OptimisationAnalaysis\PARI4_PD_20X4X2048_individual\rollingball-r10-minmax-pixelfilteirng\rollingball-r10-minmax-pixelfiltering-pipexseq-myANALYSIS\Tresholding\adata_all.h5ad"
adata_all.write(adata_save_path)
print(f"AnnData object saved at {adata_save_path}")

## Cell type assignment based on rules and thresholds

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from anndata import AnnData

# Load data from CSV
csv_path = r"C:\Users\ivasu\Desktop\PDDLBD_PARI_Pilot_OptimisationAnalaysis\PARI4_PD_20X4X2048_individual\rollingball-r10-minmax-pixelfilteirng\rollingball-r10-minmax-pixelfiltering-pipexseq-myANALYSIS\cell_data_log10_zscore.csv"
dataframe_filtered_normalized = pd.read_csv(csv_path)

# Define reliable data columns
reliable_data = ['cell_id', 'size', 'x', 'y', 'AQP4', 'CD3', 'CD31', 'CD45', 'CD163', 'Claudin5', 'FoxA2', 'GFAP', 
                 'HLA-DR', 'Ki67', 'MAP2', 'NEFL', 'Nes', 'NeuN', 'Olig2', 'P2Y12', 'PRKN', 'SNCA', 'Sox2', 'Sox9', 
                 'Syp', 'TMEM119', 'TUBB3', 'Ubiquitin', 'Vim']
dataframe_filtered_normalized = dataframe_filtered_normalized[reliable_data]

# Define metadata and marker data columns
metadata = ['cell_id', 'size', 'x', 'y']
marker_data = ['AQP4', 'CD3', 'CD31', 'CD45', 'CD163', 'Claudin5', 'FoxA2', 'GFAP', 'HLA-DR', 'Ki67', 
    'MAP2', 'NEFL', 'Nes', 'NeuN', 'Olig2', 'P2Y12', 'PRKN', 'SNCA', 'Sox2', 'Sox9', 
    'Syp', 'TMEM119', 'TUBB3', 'Ubiquitin', 'Vim']

# Extract coordinates
coordinates = np.array(dataframe_filtered_normalized[['x', 'y']])

# Prepare AnnData object
meta = dataframe_filtered_normalized[metadata]
meta.index = meta.index.astype(str)

only_num = dataframe_filtered_normalized.drop(metadata, axis=1)
only_num.columns = only_num.columns.astype(str)
adata_all = AnnData(X=only_num.values, obs=meta)
adata_all.obsm["coord_xy"] = coordinates
adata_all.var_names = marker_data
adata_all.var["markers"] = only_num.columns.tolist()

print(f"Markers specific location, mean for nucleus, max for cytoplasm and cell: {adata_all}")


# Define cell type classification rules
cell_type_rules = {
    "Neurons": ["NeuN"],
    "Oligodendrocytes": ["Olig2"],
    "Astrocytes": ["Sox9", "GFAP"],
    "Microglia": ["CD45", "TMEM119"],
    "Other Immune Cells": ["CD45"],
    "Blood Vessels": ["CD31"],
    "Tcells": ["CD45", "CD3"]
}

# Function to determine cell type
def classify_cell(row):
    positive_markers = {
        marker: intensity for marker, intensity in row.items()
        if marker in thresholds and intensity > thresholds[marker]
    }

    if not positive_markers:
        return "Unclassified"

    # Assign cell types based on rules
    cell_types = []
    for cell_type, markers in cell_type_rules.items():
        if all(marker in positive_markers for marker in markers):  # Check all required markers
            cell_types.append((cell_type, max(positive_markers[m] for m in markers)))  # Store max intensity

    if not cell_types:
        return "Unclassified"

    # Resolve conflicts: Choose the type with the highest marker intensity
    best_cell_type = max(cell_types, key=lambda x: x[1])[0]
    
    return best_cell_type

# Apply classification function to AnnData
adata_all.obs["Cell Type"] = [classify_cell(row) for row in only_num.to_dict(orient='records')]
adata_all.obs["Cell Type"] = adata_all.obs["Cell Type"].fillna("Unclassified")
adata_all.obs["Cell Type"] = adata_all.obs["Cell Type"].astype(str)

# Display results using Pandas DataFrame
print(adata_all.obs.head())

# Save AnnData object
adata_save_path = r"C:\Users\ivasu\Desktop\PDDLBD_PARI_Pilot_OptimisationAnalaysis\PARI4_PD_20X4X2048_individual\rollingball-r10-minmax-pixelfilteirng\rollingball-r10-minmax-pixelfiltering-pipexseq-myANALYSIS\Tresholding\adata_all_withGFAPforAstrocytes.h5ad"
adata_all.write(adata_save_path)
print(f"AnnData object saved at {adata_save_path}")

## Plotting (pie chart, spatial distrubution and heatmap)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

# Load AnnData object
adata_save_path = r"C:\Users\ivasu\Desktop\PDDLBD_PARI_Pilot_OptimisationAnalaysis\PARI4_PD_20X4X2048_individual\rollingball-r10-minmax-pixelfilteirng\rollingball-r10-minmax-pixelfiltering-pipexseq-myANALYSIS\Tresholding\adata_all_withGFAPforAstrocytes.h5ad"
adata_all = ad.read_h5ad(adata_save_path)
print(f"AnnData object loaded from {adata_save_path}")

# Define output directory
output_dir = os.path.join(os.path.dirname(adata_save_path), "figures")
os.makedirs(output_dir, exist_ok=True)

# Generate consistent color mapping for cell types
unique_cell_types = adata_all.obs["Cell Type"].unique()
color_palette = sns.color_palette("tab10", len(unique_cell_types))
color_map = dict(zip(unique_cell_types, color_palette))

# Count cell types
cell_type_counts = adata_all.obs["Cell Type"].value_counts()

mean_expression = adata_all.to_df().groupby(adata_all.obs["Cell Type"]).mean()

# Plot and save pie chart with consistent colors
plt.figure(figsize=(7, 7))
plt.pie(cell_type_counts, labels=cell_type_counts.index, autopct='%1.1f%%', startangle=90, 
        textprops={'fontsize': 12}, colors=[color_map[ct] for ct in cell_type_counts.index])
plt.title("Cell Type Distribution", fontsize=14)
plt.tight_layout()
pie_chart_path = os.path.join(output_dir, "cell_type_distribution.png")
plt.savefig(pie_chart_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Pie chart saved at {pie_chart_path}")

# Plot and save spatial distribution with consistent colors
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=adata_all.obs['x'], 
    y=adata_all.obs['y'], 
    hue=adata_all.obs['Cell Type'],
    palette=color_map, 
    s=10, alpha=0.8
)
plt.title("Spatial Distribution of Cells", fontsize=14)
plt.xlabel("X Coordinate", fontsize=12)
plt.ylabel("Y Coordinate", fontsize=12)
plt.gca().set_aspect('equal', adjustable='datalim')
plt.gca().invert_yaxis()
plt.tight_layout()
spatial_plot_path = os.path.join(output_dir, "spatial_distribution.png")
plt.savefig(spatial_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Spatial plot saved at {spatial_plot_path}")

# Generate and save heatmap of marker expression per cell type with better readability
plt.figure(figsize=(14, 8))
sns.heatmap(mean_expression, cmap="viridis", annot=True, fmt=".2f", annot_kws={"size": 10})
plt.title("Marker Expression per Cell Type", fontsize=14)
plt.xlabel("Markers", fontsize=12)
plt.ylabel("Cell Type", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
heatmap_path = os.path.join(output_dir, "marker_expression_heatmap.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Heatmap saved at {heatmap_path}")
